# Phase 4 — IMU Integration with VGGT

Evaluates whether initialising / constraining VGGT camera poses with IMU
gyroscope pre-integration improves 3-D reconstruction accuracy.

**Dataset**: TUM Visual-Inertial Dataset — `room1` sequence (40 frames sampled uniformly)  
**Reference**: https://cvg.cit.tum.de/data/datasets/visual-inertial-dataset

**Experiment**
1. Download TUM-VI `room1` (EuRoC 512-px export).
2. Sub-sample to ≤ 40 frames.
3. Run baseline VGGT (vision-only).
4. Integrate gyroscope between image frames.
5. Fuse VGGT poses with IMU rotations at α ∈ {0, 0.25, 0.5, 0.75, 1.0}.
6. Evaluate ATE / RPE vs. motion-capture ground truth.
7. Visualise trajectories.

## 0  — Setup

In [ ]:
# ── Cell 0-A: Install dependencies (run once, kernel will auto-restart) ──────
import subprocess, sys, os

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *args])

# 1. Pin numpy first to avoid ABI mismatch (Kaggle pre-installs an older
#    numpy ABI; upgrading before everything else prevents the
#    "dtype size changed" ValueError when pandas/scipy are imported).
pip("--upgrade", "numpy")

# 2. Other dependencies
pip("plotly", "pandas", "pillow", "scipy")

# 3. VGGT
if not os.path.isdir("vggt"):
    subprocess.check_call(["git", "clone", "--depth", "1",
                           "https://github.com/facebookresearch/vggt.git"])
    pip("-e", "vggt")

# 4. vggt-eval (this repo)
if not os.path.isdir("vggt-eval"):
    subprocess.check_call(["git", "clone", "--depth", "1",
                           "https://github.com/insha-py/vggt-eval.git"])

sys.path.insert(0, "vggt-eval")

# 5. Restart the kernel so the freshly-installed numpy ABI is loaded.
#    All subsequent cells run normally after the restart.
print("Packages installed. Restarting kernel to load updated numpy …")
try:
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)
except Exception:
    print("Auto-restart unavailable — please restart the kernel manually "
          "(Run > Restart & Clear Output), then start from Cell 1.")

In [ ]:
import os, sys, json

# Re-add paths after kernel restart (setup cell's sys.path doesn't survive restart)
for p in ["vggt-eval", "vggt"]:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd          # must come before plotly.express
import torch
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from src.pipeline    import VGGTPipeline, load_images_from_list, run_vggt_inference
from src.tum_vi      import TUMVIDataset
from src.imu         import IMUPreintegrator, estimate_gyro_bias
from src.imu_fusion  import IMUVGGTFusion, alpha_sweep
from src.metrics     import compute_ate, compute_rpe, compute_rotation_errors, print_metrics_table

# ---- Reproducibility ----
np.random.seed(42)
torch.manual_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 1  — Download & Load TUM-VI `room1`

In [ ]:
SEQUENCE   = "room1"   # change to room2…6 or corridor1…5 to try other scenes
N_FRAMES   = 40        # keep ≤ 50
DOWNLOAD_DIR = "/tmp/tumvi"   # Colab/Kaggle tmp; use a Drive path for persistence

ds = TUMVIDataset(sequence=SEQUENCE, n_frames=N_FRAMES, download_dir=DOWNLOAD_DIR)
ds.download()    # idempotent — skips if already extracted
data = ds.load()

images_np        = data["images_np"]          # (N, 3, H, W)
image_paths      = data["image_paths"]
image_timestamps = data["image_timestamps"]   # seconds
imu_readings     = data["imu_readings"]
gt_extrinsics    = data["gt_extrinsics"]      # (N, 3, 4) world-to-body
calib            = data["calib"]

N, _, H, W = images_np.shape
print(f"\nLoaded {N} frames  ({H}×{W} px)")
print(f"IMU readings : {len(imu_readings)}")
print(f"GT extrinsics: {gt_extrinsics.shape}")
print(f"Time span    : {image_timestamps[-1] - image_timestamps[0]:.2f} s")

In [ ]:
# Quick thumbnail grid of selected frames
import matplotlib.pyplot as plt

n_show = min(8, N)
fig, axes = plt.subplots(1, n_show, figsize=(2 * n_show, 2))
for ax, idx in zip(axes, np.linspace(0, N - 1, n_show, dtype=int)):
    ax.imshow(images_np[idx].transpose(1, 2, 0))
    ax.axis("off")
    ax.set_title(f"#{idx}", fontsize=8)
plt.suptitle(f"TUM-VI {SEQUENCE} — selected frames", y=1.02)
plt.tight_layout()
plt.show()

## 2  — IMU Pre-integration

In [ ]:
# Estimate gyroscope bias from first second of stationary data
gyro_bias = estimate_gyro_bias(imu_readings, duration_s=1.0)
print(f"Gyroscope bias: {gyro_bias}  (rad/s)")

calib.gyro_bias = gyro_bias

integrator = IMUPreintegrator(calibration=calib)

# Gyroscope-only integration → one absolute rotation per image frame
R_imu_abs = integrator.gyro_only_rotations(
    imu_readings, image_timestamps, R0=np.eye(3)
)  # (N, 3, 3)
print(f"IMU rotations computed for {len(R_imu_abs)} frames")

# Show rotation angles relative to first frame
from src.imu import so3_log
rot_angles_deg = []
for i in range(N):
    R_rel = R_imu_abs[i] @ R_imu_abs[0].T
    angle = np.degrees(np.linalg.norm(so3_log(R_rel)))
    rot_angles_deg.append(angle)

fig = px.line(
    x=image_timestamps, y=rot_angles_deg,
    labels={"x": "Time (s)", "y": "Rotation from frame 0 (°)"},
    title="IMU integrated rotation magnitude",
)
fig.show()

In [ ]:
# Also show gyro + accel signals over the selected time window
t_start = image_timestamps[0] - 0.1
t_end   = image_timestamps[-1] + 0.1
imu_window = [r for r in imu_readings if t_start <= r.timestamp <= t_end]

imu_t     = [r.timestamp - image_timestamps[0] for r in imu_window]
gyro_xyz  = np.array([r.gyro  for r in imu_window])
accel_xyz = np.array([r.accel for r in imu_window])

fig = make_subplots(rows=2, cols=1,
                    subplot_titles=("Gyroscope (rad/s)", "Accelerometer (m/s²)"))
for i, label in enumerate(["x", "y", "z"]):
    fig.add_trace(go.Scatter(x=imu_t, y=gyro_xyz[:, i],
                             name=f"gyro_{label}", line=dict(width=1)), row=1, col=1)
    fig.add_trace(go.Scatter(x=imu_t, y=accel_xyz[:, i],
                             name=f"accel_{label}", line=dict(width=1)), row=2, col=1)
fig.update_xaxes(title_text="Time from sequence start (s)")
fig.update_layout(height=500, title_text="IMU signals over selected window")
fig.show()

## 3  — Baseline VGGT (vision-only)

In [ ]:
# Load VGGT model
pipe = VGGTPipeline()
pipe.load_model()

In [ ]:
import torch.nn.functional as F

VGGT_RES = 518   # model's native input resolution

# Resize images to VGGT's expected resolution
images_t = torch.from_numpy(images_np)                       # (N, 3, H, W)
images_t = F.interpolate(images_t, size=(VGGT_RES, VGGT_RES),
                          mode="bilinear", align_corners=False)

print(f"Running VGGT on {N} frames at {VGGT_RES}×{VGGT_RES} …")

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

import time
t0 = time.perf_counter()
vggt_raw = run_vggt_inference(pipe.model, images_t, pipe.device, pipe.dtype)
if torch.cuda.is_available():
    torch.cuda.synchronize()
vggt_time = time.perf_counter() - t0

vggt_ext = vggt_raw["extrinsic"]   # (N, 3, 4)
vggt_int = vggt_raw["intrinsic"]   # (N, 3, 3)

peak_mb = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0
print(f"VGGT inference: {vggt_time:.2f} s   peak GPU: {peak_mb:.0f} MB")

In [ ]:
# Baseline metrics (vision-only VGGT)
ate_baseline  = compute_ate(vggt_ext, gt_extrinsics, align=True, with_scale=True)
rpe_baseline  = compute_rpe(vggt_ext, gt_extrinsics, step=1)
rot_baseline  = compute_rotation_errors(vggt_ext, gt_extrinsics)

print_metrics_table(ate_baseline, "Baseline VGGT — ATE")
print_metrics_table(rpe_baseline, "Baseline VGGT — RPE")
print_metrics_table(rot_baseline, "Baseline VGGT — Rotation Error")

## 4  — VGGT + IMU Fusion (alpha sweep)

In [ ]:
ALPHAS = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]

print("Alpha sweep (0=pure VGGT, 1=pure IMU):")
sweep_results = alpha_sweep(
    vggt_extrinsics   = vggt_ext,
    imu_readings      = imu_readings,
    image_timestamps  = image_timestamps,
    gt_extrinsics     = gt_extrinsics,
    alphas            = ALPHAS,
    calibration       = calib,
)

df_sweep = pd.DataFrame(sweep_results)
print("\nSweep table:")
print(df_sweep.to_string(index=False, float_format="{:.4f}".format))

In [ ]:
# Plot ATE vs alpha
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_sweep["alpha"], y=df_sweep["ate_mean"],
    mode="lines+markers", name="ATE mean",
    line=dict(color="royalblue", width=2),
    error_y=dict(type="data", array=df_sweep["ate_rmse"] - df_sweep["ate_mean"],
                 visible=True),
))
fig.add_trace(go.Scatter(
    x=df_sweep["alpha"], y=df_sweep["ate_median"],
    mode="lines+markers", name="ATE median",
    line=dict(color="darkorange", width=2, dash="dot"),
))
# Mark baseline (alpha=0) with a dashed horizontal reference
baseline_ate = df_sweep[df_sweep["alpha"] == 0.0]["ate_mean"].values
if len(baseline_ate):
    fig.add_hline(y=float(baseline_ate[0]), line_dash="dash",
                  line_color="grey", annotation_text="VGGT baseline")

fig.update_layout(
    title=f"ATE vs IMU blend weight α — TUM-VI {SEQUENCE} ({N} frames)",
    xaxis_title="α  (0 = pure VGGT,  1 = pure IMU)",
    yaxis_title="ATE (m, after Umeyama alignment)",
    legend=dict(x=0.6, y=0.95),
)
fig.show()

In [ ]:
# RPE translation and rotation vs alpha
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("RPE Translation (m)", "RPE Rotation (°)"))
fig.add_trace(go.Scatter(x=df_sweep["alpha"], y=df_sweep["rpe_trans"],
                          mode="lines+markers", name="RPE trans",
                          line=dict(color="royalblue")), row=1, col=1)
fig.add_trace(go.Scatter(x=df_sweep["alpha"], y=df_sweep["rpe_rot"],
                          mode="lines+markers", name="RPE rot",
                          line=dict(color="firebrick")), row=1, col=2)
fig.update_xaxes(title_text="α")
fig.update_layout(height=400, title_text="RPE vs α", showlegend=False)
fig.show()

## 5  — Best-alpha Trajectory vs Ground Truth

In [ ]:
# Find alpha with lowest ATE
best_row   = df_sweep.loc[df_sweep["ate_mean"].idxmin()]
best_alpha = float(best_row["alpha"])
print(f"Best alpha: {best_alpha}  (ATE mean = {best_row['ate_mean']:.4f} m)")

# Compute fused extrinsics at best alpha
fuser_best = IMUVGGTFusion(alpha=best_alpha, calibration=calib)
fused_best = fuser_best.fuse(vggt_ext, imu_readings, image_timestamps)  # (N, 3, 4)

# Also at alpha=0.5 for comparison
fuser_05 = IMUVGGTFusion(alpha=0.5, calibration=calib)
fused_05 = fuser_05.fuse(vggt_ext, imu_readings, image_timestamps)

In [ ]:
def camera_centers(ext: np.ndarray) -> np.ndarray:
    """Extract (N,3) camera centres from (N,3,4) extrinsics."""
    R = ext[:, :3, :3]
    t = ext[:, :3,  3]
    return -np.einsum("nij,nj->ni", R.transpose(0, 2, 1), t)

# Camera centres
c_vggt   = camera_centers(vggt_ext)
c_fused  = camera_centers(fused_best)
c_fused5 = camera_centers(fused_05)

# GT positions (world frame, from mocap)
# gt_extrinsics stores position directly in [:, :3, 3]
c_gt = gt_extrinsics[:, :3, 3]

# Umeyama align VGGT to GT for fair visualisation
from src.metrics import align_trajectories_umeyama, apply_pose_alignment

R_al, t_al, s_al = align_trajectories_umeyama(c_vggt, c_gt, with_scale=True)

def align_centers(c):
    return s_al * (R_al @ c.T).T + t_al

c_vggt_al  = align_centers(c_vggt)
c_fused_al = align_centers(c_fused)
c_fus5_al  = align_centers(c_fused5)

# 3-D trajectory plot
fig = go.Figure()

def traj_trace(c, name, color, dash="solid"):
    return go.Scatter3d(
        x=c[:, 0], y=c[:, 1], z=c[:, 2],
        mode="lines+markers",
        name=name,
        line=dict(color=color, width=3, dash=dash),
        marker=dict(size=3, color=color),
    )

fig.add_trace(traj_trace(c_gt,        "Ground truth",             "black"))
fig.add_trace(traj_trace(c_vggt_al,   "VGGT baseline (α=0)",      "royalblue"))
fig.add_trace(traj_trace(c_fus5_al,   "VGGT+IMU (α=0.5)",         "darkorange", "dot"))
fig.add_trace(traj_trace(c_fused_al,  f"VGGT+IMU (α={best_alpha})", "firebrick", "dash"))

fig.update_layout(
    title=f"Camera trajectory — TUM-VI {SEQUENCE}",
    scene=dict(
        xaxis_title="X (m)", yaxis_title="Y (m)", zaxis_title="Z (m)",
        aspectmode="data",
    ),
    legend=dict(x=0.0, y=1.0),
)
fig.show()

In [ ]:
# Per-frame ATE plot
from src.metrics import _camera_centers

def per_frame_ate(pred_ext, gt_ext):
    pred_c = _camera_centers(pred_ext)
    gt_c   = _camera_centers(gt_ext)
    R, t, s = align_trajectories_umeyama(pred_c, gt_c, with_scale=True)
    pred_c_al = s * (R @ pred_c.T).T + t
    return np.linalg.norm(pred_c_al - gt_c, axis=1)

err_vggt   = per_frame_ate(vggt_ext,   gt_extrinsics)
err_fused5 = per_frame_ate(fused_05,   gt_extrinsics)
err_fused  = per_frame_ate(fused_best, gt_extrinsics)

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(N)), y=err_vggt,
                          name="VGGT baseline", line=dict(color="royalblue")))
fig.add_trace(go.Scatter(x=list(range(N)), y=err_fused5,
                          name="VGGT+IMU α=0.5", line=dict(color="darkorange", dash="dot")))
fig.add_trace(go.Scatter(x=list(range(N)), y=err_fused,
                          name=f"VGGT+IMU α={best_alpha}", line=dict(color="firebrick", dash="dash")))
fig.update_layout(
    title="Per-frame ATE",
    xaxis_title="Frame index",
    yaxis_title="ATE (m)",
)
fig.show()

## 6  — Rotation Error Analysis

In [ ]:
from src.metrics import _rotation_angle_deg

def per_frame_rot_error(pred_ext, gt_ext):
    errors = []
    for p, g in zip(pred_ext, gt_ext):
        R_err = p[:3, :3] @ g[:3, :3].T
        errors.append(_rotation_angle_deg(R_err))
    return np.array(errors)

rot_err_vggt   = per_frame_rot_error(vggt_ext,   gt_extrinsics)
rot_err_fused5 = per_frame_rot_error(fused_05,   gt_extrinsics)
rot_err_fused  = per_frame_rot_error(fused_best, gt_extrinsics)

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(N)), y=rot_err_vggt,
                          name="VGGT baseline", line=dict(color="royalblue")))
fig.add_trace(go.Scatter(x=list(range(N)), y=rot_err_fused5,
                          name="VGGT+IMU α=0.5", line=dict(color="darkorange", dash="dot")))
fig.add_trace(go.Scatter(x=list(range(N)), y=rot_err_fused,
                          name=f"VGGT+IMU α={best_alpha}", line=dict(color="firebrick", dash="dash")))
fig.update_layout(
    title="Per-frame Rotation Error",
    xaxis_title="Frame index",
    yaxis_title="Rotation error (°)",
)
fig.show()

print("\nRotation error summary:")
for name, errs in [("VGGT", rot_err_vggt),
                   ("Fused α=0.5", rot_err_fused5),
                   (f"Fused α={best_alpha}", rot_err_fused)]:
    print(f"  {name:<18}  mean={errs.mean():.2f}°  median={np.median(errs):.2f}°  max={errs.max():.2f}°")

## 7  — Summary Table

In [ ]:
def full_metrics(ext, label):
    ate = compute_ate(ext, gt_extrinsics, align=True, with_scale=True)
    rpe = compute_rpe(ext, gt_extrinsics, step=1)
    rot = compute_rotation_errors(ext, gt_extrinsics)
    return dict(
        Method      = label,
        ATE_mean    = round(ate["mean"],   4),
        ATE_rmse    = round(ate["rmse"],   4),
        ATE_median  = round(ate["median"], 4),
        RPE_t_mean  = round(rpe["trans_mean"],  4),
        RPE_r_mean  = round(rpe["rot_mean"],    2),
        Rot_mean_deg= round(rot["mean"],   2),
    )

rows = [
    full_metrics(vggt_ext,   f"VGGT baseline (α=0)"),
    full_metrics(fused_05,    "VGGT+IMU α=0.5"),
    full_metrics(fused_best,  f"VGGT+IMU α={best_alpha} (best)"),
]

df_summary = pd.DataFrame(rows)
print(f"\n=== Results: TUM-VI {SEQUENCE}  ({N} frames) ===")
print(df_summary.to_string(index=False))

# Save to CSV
out_dir = "results"
os.makedirs(out_dir, exist_ok=True)
df_summary.to_csv(f"{out_dir}/phase4_imu_{SEQUENCE}.csv", index=False)
df_sweep.to_csv(f"{out_dir}/phase4_alpha_sweep_{SEQUENCE}.csv", index=False)
print(f"\nResults saved to {out_dir}/")

## 8  — Point Cloud Comparison (optional, GPU memory permitting)

In [ ]:
# Generate point clouds with VGGT depth maps using both extrinsics
from src.pipeline import depth_to_point_cloud

pts_vggt, clr_vggt = depth_to_point_cloud(
    vggt_raw["depth_map"], vggt_raw["depth_conf"],
    vggt_ext, vggt_int,
    images_rgb=images_t.numpy(),
    conf_thresh=5.0,
)
pts_fused, clr_fused = depth_to_point_cloud(
    vggt_raw["depth_map"], vggt_raw["depth_conf"],
    fused_best, vggt_int,
    images_rgb=images_t.numpy(),
    conf_thresh=5.0,
)
print(f"VGGT point cloud : {len(pts_vggt):,} points")
print(f"Fused point cloud: {len(pts_fused):,} points")

In [ ]:
# Sub-sample for Plotly (max ~50k points for browser)
MAX_VIS = 40_000

def subsample_pc(pts, clrs, n=MAX_VIS):
    if len(pts) > n:
        idx = np.random.choice(len(pts), n, replace=False)
        return pts[idx], clrs[idx] if clrs is not None else None
    return pts, clrs

pts_v, clr_v = subsample_pc(pts_vggt,  clr_vggt)
pts_f, clr_f = subsample_pc(pts_fused, clr_fused)

def pc_trace(pts, clrs, name):
    if clrs is not None:
        col = [f"rgb({r},{g},{b})" for r, g, b in clrs]
    else:
        col = "grey"
    return go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode="markers",
        marker=dict(size=1, color=col),
        name=name,
    )

fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}]],
                    subplot_titles=("VGGT baseline", f"VGGT+IMU α={best_alpha}"))
fig.add_trace(pc_trace(pts_v, clr_v, "VGGT"),  row=1, col=1)
fig.add_trace(pc_trace(pts_f, clr_f, "Fused"), row=1, col=2)
fig.update_layout(
    height=600,
    title_text=f"Point cloud comparison — TUM-VI {SEQUENCE}",
)
fig.show()

## Discussion

**What the numbers show:**

- `α = 0` is pure VGGT.  As α increases we blend in more IMU rotation.
- If the best α is **small (< 0.3)**: VGGT's rotation estimates are already
  good; IMU adds little.
- If the best α is **large (> 0.5)**: VGGT struggles with rotation (often
  the case for fast rotational motion), and IMU integration corrects it.
- RPE rotation error tends to decrease with IMU blending for fast-rotation
  sequences because the gyroscope is more accurate than appearance matching
  for inter-frame rotation when frames are far apart in angle.

**Limitations of this simple fusion:**

- Translation is purely from VGGT (no IMU position integration).
- No loop-closure or global consistency enforcement.
- Camera-IMU extrinsic calibration uses approximate defaults;
  reading `sensor.yaml` would improve accuracy.
- A proper VIO (Visual-Inertial Odometry) system would jointly optimise
  poses, IMU biases, and 3-D structure.

**Next steps:**

- Try accelerometer-aided position integration (requires careful gravity
  alignment and bias estimation).
- Evaluate on other sequences (room2–6, corridor) with different motion profiles.
- Implement sliding-window (Phase 3 `SlidingWindowProcessor`) with
  IMU-aided chunk alignment instead of Procrustes.